In [1]:
from pathlib import Path
import pymupdf

## Load & Inspect

- **Documents:** 1 PDF file divided into 7 books
- **Pages:** PDF File
- **needing OCR:** Pages with images 
- **books**
    1. Harry Potter and the Sorcerer Stone
    2. Harry Potter and the Chamber of Secrets
    3. Harry Potter and the Prisoner of Azkaban
    4. Harry Potter and the Goblet of Fire
    5. Harry Potter and the Order of the Phoenix
    6. Harry Potter and the Half-Blood Prince
    7. Harry Potter and the Deathly Hallows

In [2]:
pdf_path =Path("../document/harrypotter.pdf")
markdown_path=Path("content.md")

markdown_file=[]

#read pdf file
with pymupdf.open(pdf_path) as pdf:
    print("Number of pages:", len(pdf),"\n")

    empty_pages = 0
    image_pages = 0

    for page_number , page in enumerate(pdf,start=1):
        text=page.get_text().strip()
        images = page.get_images(full=True)

        if not text and not images:
            empty_pages +=1

        elif not text and images:
            image_pages +=1

        
        markdown_file.append(f"## Page {page_number}\n\n")
        markdown_file.append(text)
        markdown_file.append("\n\n")  


    print(f"number of empty pages: {empty_pages}")
    print(f"number of image pages: {image_pages}")
    print("="*40)

with open(markdown_path, "w", encoding="utf-8") as file:
        file.write("".join(markdown_file))



Number of pages: 3623 

number of empty pages: 9
number of image pages: 10


## Chunking Strategy 

**I used semantic chunking for the documents.**
- First I split the text into sentences then I used sentence embeddings to calculate the similarity between neighboring sentences
- If the similarity is above threshold -> **0.78** I merge the sentences into the same chunk

- I set the maximum number of sentences each chunk can contain **from 1 to 40** sentences to avoid creating very large chunks
- I chose the semantic chunking approach because i want sentences from the same topic to be grouped together Instead of using fixed-size chunks 
which may split a topic even if the idea is not finished
- I made sure that sentences from the same book are compared with each other but sentences from different books are not

In [3]:
import re
import nltk
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt_tab")

model_transformer =SentenceTransformer("intfloat/multilingual-e5-small", device="cuda")


BOOK_RANGES = [
    ("Harry Potter and the Sorcerer Stone", 12, 274),
    ("Harry Potter and the Chamber of Secrets", 282, 565),
    ("Harry Potter and the Prisoner of Azkaban", 573, 939),
    ("Harry Potter and the Goblet of Fire", 949, 1560),
    ("Harry Potter and the Order of the Phoenix", 1570, 2406),
    ("Harry Potter and the Half-Blood Prince", 2409, 2964),
    ("Harry Potter and the Deathly Hallows", 2974, 3622),
]


def get_book_name(page_number):
    for book_name, first_page, last_page in BOOK_RANGES:
        if first_page <= page_number <= last_page:
            return book_name

    return None


#remove extra spaces and new lines
def clean_text(text):
    text=re.sub(r"\s+"," ",text)
    text=re.sub(r'[\r\n]+', '\n', text).strip().lower()
    return text


text=markdown_path.read_text(encoding="utf-8")

full_text=re.split(r"^##\s*Page\s+(\d+)\s*$",text, flags=re.MULTILINE)

chunks=[]
all_sentances=[]

for i in range(1,len(full_text),2):
    page_number=int(full_text[i])
    page_content=clean_text(full_text[i+1])
    book_name=get_book_name(page_number)


    if book_name:
        sentences = sent_tokenize(page_content)
        for sen in sentences:
           all_sentances.append({
               "text": sen,
               "page": page_number,
               "book_name":book_name
           }) 
        
texts=[s["text"] for s in all_sentances]

current_chunk={
    "text":all_sentances[0]["text"],
    "page":[all_sentances[0]["page"]],
    "book_name":[all_sentances[0]["book_name"]],
    "sentance_count":1
}

embeddings= model_transformer.encode(texts,normalize_embeddings=True, show_progress_bar=True).tolist()

threshold=0.78
max_sentences=40
similarities=[]

def is_full():
    return current_chunk["sentance_count"] < max_sentences


for i in range(len(embeddings)-1):
    similarity =cosine_similarity(
        [embeddings[i]],
        [embeddings[i+1]]
    ).item()

    similarities.append(similarity)

    next =all_sentances[i+1]

    #don't merge between two diffrenent books 
    if similarity >= threshold and next["book_name"] == current_chunk["book_name"][-1] and is_full():
        current_chunk["sentance_count"] +=1
        current_chunk["text"] += " "+next["text"]

        if next["page"] not in current_chunk["page"]:
            current_chunk["page"].append(next["page"])


    else:
        chunks.append(current_chunk)

        current_chunk={
            "text":next["text"],
            "page":[next["page"]],
            "book_name":[next["book_name"]],
            "sentance_count":1
            }

chunks.append(current_chunk)
print("Chuncks are ready ")


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Mohamed\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/3331 [00:00<?, ?it/s]

Chuncks are ready 


In [4]:
#number of chunks
print(f"#number of chunks: {len(chunks)}")

#number of chunks: 11530


In [5]:
import numpy as np
# Similarity distribution
print("25%:", np.percentile(similarities, 25))
print("50%:", np.percentile(similarities, 50))
print("75%:", np.percentile(similarities, 75))
print("90%:", np.percentile(similarities, 90))

25%: 0.7974968349823348
50%: 0.819253012833516
75%: 0.8452144180973812
90%: 0.9422386146228463


In [6]:
# get the three largest chunks by number of grouped sentences
largest_chunks = sorted(
    chunks,
    key=lambda x: x["sentance_count"],
    reverse=True
)[:3]

for i, chunk in enumerate(largest_chunks, 1):
    print(f"===== Chunk {i} =====")
    print("Sentence count:", chunk["sentance_count"])
    print("Pages:", chunk["page"])
    print("Book:", chunk["book_name"])
    print("Text:", chunk["text"])

===== Chunk 1 =====
Sentence count: 40
Pages: [29, 30]
Book: ['Harry Potter and the Sorcerer Stone']
Text: said aunt petunia, looking furiously at harry as though he’d planned this. harry knew he ought to feel sorry that mrs. figg had broken her leg, but it wasn’t easy when he reminded himself it would be a whole year before he had to look at tibbles, snowy, mr. paws, and tufty again. “we could phone marge,” uncle vernon suggested. “don’t be silly, vernon, she hates the boy.” the dursleys often spoke about harry like this, as though he wasn’t there — or rather, as though he was something very nasty that couldn’t understand them, like a slug. “what about what’s-her-name, your friend — yvonne?” “on vacation in majorca,” snapped aunt petunia. “you could just leave me here,” harry put in hopefully (he’d be able to watch what he wanted on television for a change and maybe even have a go on dudley’s computer). aunt petunia looked as though she’d just swallowed a lemon. “and come back and fin

## Embeddings & Vector Store 

In [40]:
import chromadb

client = chromadb.PersistentClient(path="../backend/data/vector_store")
collection = client.create_collection(name="Harry_Potter",configuration={"hnsw": {"space": "cosine"}})

texts = [f"passage: {chunk['text']}" for chunk in chunks]
# Get embeddings for each chunk 
embeddings= model_transformer.encode(texts,normalize_embeddings=True, show_progress_bar=True).tolist()

batch_size=5000

for start in range(0,len(chunks),batch_size):
        end= min(start + batch_size, len(chunks))
        collection.add(
            ids=[f"chunk_{i+1}" for i in range(start, end)],
            embeddings=embeddings[start:end],

            documents=[chunk["text"] for chunk in chunks[start:end]],
            metadatas=[{
                "book_name": ", ".join(chunk["book_name"]),
                "pages_number": ", ".join(map(str, chunk["page"]))}
                    for chunk in chunks[start: end] ]
        )
print(f"{len(chunks)} has been stored into backend/data/vector_store")

Batches:   0%|          | 0/361 [00:00<?, ?it/s]

11530 has been stored into backend/data/vector_store


#### Query Route 
- The goal is to determine the type of query before generating its embedding whether it is related to the system and requires retrieval, is casual conversation or is a question outside the system's scope.

In [65]:
import ollama

def query_router(query):
    system_prompt = """
you are a query router for a Harry Potter RAG system.
you should understand the knowladge base weil and then classify it.

Classify the user's query into exactly one label:
    retrieve:  any question related to the Harry Potter books, including characters, events, places, objects, or story details.

    chat: casual conversation, greetings, thanks, or general conversation that does not require the Harry Potter knowledge base. 

    off_topic: The question is unrelated to Harry Potter and is not casual conversation.
Return exactly one label:
retrieve
chat
off_topic
"""

    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": query
            }
        ],
        options={
            "temperature": 0
        }
    )

    route = response["message"]["content"].strip().lower()

    if route not in ["retrieve", "chat", "off_topic"]:
        route = "off_topic"

    return route

In [76]:
def retrieve(query,top_k=3):

    embedding= model_transformer.encode(
        [f"query: {query}"],
        normalize_embeddings=True,
    ).tolist()

    results=collection.query(
        query_embeddings= embedding,
        n_results=top_k
    )

    context=""
    for i in range(top_k):
        context +=(
           f'id: {results["ids"][0][i]}\n'
           f'book_name: {results["metadatas"][0][i]["book_name"]}\n'
           f'pages: {results["metadatas"][0][i]["pages_number"]}\n'
           f'text: {results["documents"][0][i]}\n'
           '=========================================================\n'
        )
    return context,results['ids'][0]

In [77]:
questions=[{
    "query": "What was Mr. Dursley's appearance like?",
    "true_chunk":"chunk_3"
},
{
   "query": "How did Mrs. Dursley spend her day?",
   "true_chunk":"chunk_12" 
},
{
   "query": "What did Mr. Dursley think of the name Harry?",
   "true_chunk":"chunk_26" 
},
{
   "query": "Where was Dumbledore's own scar located?",
   "true_chunk":"chunk_50" 
},
{
   "query": "Where did Harry sleep?",
   "true_chunk":"chunk_61" 
},
{
   "query": "Why was Harry happy about spending the day with Dudley and Piers?",
   "true_chunk":"chunk_76" 
},
{
   "query": "What happened to Mr. Dursley's sister after she received a letter?",
   "true_chunk":"chunk_201" 
},
{
   "query": "Why did Hermione warn Harry not to wander around the school at night?",
   "true_chunk":"chunk_531" 
},
{
   "query": "Where did Hagrid keep the large black egg?",
   "true_chunk":"chunk_756" 
},
{
   "query": "What was happening to the car as it flew toward the lake?",
   "true_chunk":"chunk_1235" 
},
]
print(f"Number of questions: {len(questions)}")

Number of questions: 10


In [79]:
for q in questions:
    print(query_router(q["query"]),f"{q['query']}: true chunk {q['true_chunk']}")
    print(retrieve(q["query"])[0],"\n\n")

retrieve What was Mr. Dursley's appearance like?: true chunk chunk_3
id: chunk_7
book_name: Harry Potter and the Sorcerer Stone
pages: 14
text: the traffic moved on and a few minutes later, mr. dursley arrived in the grunnings parking lot, his mind back on drills. mr. dursley always sat with his back to the window in his office on the ninth floor. if he hadn’t, he might have found it harder to concentrate on drills that morning. he didn’t see the owls swooping past in broad daylight, though people down in the street did; they pointed and gazed open-mouthed as owl after owl sped overhead. most of them had never seen an owl even at nighttime. mr. dursley, however, had a perfectly normal, owl-free morning.
id: chunk_6
book_name: Harry Potter and the Sorcerer Stone
pages: 13, 14
text: mr. dursley blinked and stared at the cat. it stared back. as mr. dursley drove around the corner and up the road, he watched the cat in his mirror. it was now reading the sign that said privet drive — no, lo

In [80]:
def prompt_template(query,context):
    system_prompt="""
    you are an helpful assistant for Harry Potter RAG system.
    your task is answering the user's questions using only the information contained in retrieved context.
    If the answer is not there, say you do not know.

    rules: 
        Do not use outside knowledge.
        If the answer is not there, say you do not know.
        Be concise and answer directly.
        At the end of the answer, include the source information exactly as provided in the context:
        [chunk_id, book_name, pages]
    """

    user_prompt=f"Context: {context}\nQuestion: {query}"
    return system_prompt,user_prompt

## Evaluation

In [81]:
top_k = 3

precision_scores = []
recall_scores = []

for q in questions:

    query = q["query"]
    true_chunk = q["true_chunk"]

    _ , retrieved_chunks = retrieve(query, top_k)

    retrieved_chunks = set(retrieved_chunks)
    true_chunks = {true_chunk}

    #get intersection 
    retrieved_and_true = retrieved_chunks & true_chunks 

    precision = (
        len(retrieved_and_true) / len(retrieved_chunks)
        if retrieved_chunks else 0
    )

    recall = (
        len(retrieved_and_true) / len(true_chunks)
        if true_chunks else 0
    )

    precision_scores.append(precision)
    recall_scores.append(recall)

    print("=" * 80)
    print("Question:", query)
    print("True chunk:", true_chunk)
    print("Retrieved chunks:", retrieved_chunks)
    print("Precision:", precision)
    print("Recall:", recall)

Question: What was Mr. Dursley's appearance like?
True chunk: chunk_3
Retrieved chunks: {'chunk_6', 'chunk_7', 'chunk_27'}
Precision: 0.0
Recall: 0.0
Question: How did Mrs. Dursley spend her day?
True chunk: chunk_12
Retrieved chunks: {'chunk_12', 'chunk_7', 'chunk_27'}
Precision: 0.3333333333333333
Recall: 1.0
Question: What did Mr. Dursley think of the name Harry?
True chunk: chunk_26
Retrieved chunks: {'chunk_26', 'chunk_3445', 'chunk_3431'}
Precision: 0.3333333333333333
Recall: 1.0
Question: Where was Dumbledore's own scar located?
True chunk: chunk_50
Retrieved chunks: {'chunk_11419', 'chunk_50', 'chunk_8134'}
Precision: 0.3333333333333333
Recall: 1.0
Question: Where did Harry sleep?
True chunk: chunk_61
Retrieved chunks: {'chunk_799', 'chunk_961', 'chunk_2786'}
Precision: 0.0
Recall: 0.0
Question: Why was Harry happy about spending the day with Dudley and Piers?
True chunk: chunk_76
Retrieved chunks: {'chunk_76', 'chunk_118', 'chunk_80'}
Precision: 0.3333333333333333
Recall: 1.0


In [86]:
avg_recall=sum(recall_scores)/len(recall_scores)
avg_precision=sum(precision_scores)/len(precision_scores)
print(f"avrage percision score: {avg_precision}")
print(f"avrage recall score: {avg_recall}")

avrage percision score: 0.19999999999999998
avrage recall score: 0.6


In [83]:
def get_answer(query,top_k):

    context,_=retrieve(query,top_k)

    system_prompt,user_prompt=prompt_template(query,context)
    answer=ollama.chat(
        model="llama3.2:3b",
        messages=[
                    {
                        "role": "system",
                        "content": system_prompt
                    },
                    {
                        "role": "user",
                        "content": user_prompt
                    }
                ],
                options={
                    "temperature": 0
                }
            )
        
    return answer["message"]["content"]

In [90]:
import pandas as pd

evaluation_results = []

for question in questions:
    query = question["query"]
    true_chunk = question["true_chunk"]

    _, retrieved_chunks = retrieve(query, 3)
    answer = get_answer(query, 3)

    correct = true_chunk in retrieved_chunks

    evaluation_results.append({
        "Question": query,
        "Retrieved Chunks": ", ".join(retrieved_chunks),
        "Answer": answer,
        "Correct": correct
    })

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,Question,Retrieved Chunks,Answer,Correct
0,What was Mr. Dursley's appearance like?,"chunk_7, chunk_6, chunk_27",I don't know.,False
1,How did Mrs. Dursley spend her day?,"chunk_27, chunk_12, chunk_7","Mrs. Dursley had a nice, normal day. She told ...",True
2,What did Mr. Dursley think of the name Harry?,"chunk_26, chunk_3431, chunk_3445","Mr. Dursley thought the name Harry was ""nasty,...",True
3,Where was Dumbledore's own scar located?,"chunk_50, chunk_8134, chunk_11419",Dumbledore's own scar was located above his le...,True
4,Where did Harry sleep?,"chunk_961, chunk_799, chunk_2786",Harry slept in a bed with white linen sheets i...,False
5,Why was Harry happy about spending the day wit...,"chunk_118, chunk_80, chunk_76",Harry was happy about spending the day with Du...,True
6,What happened to Mr. Dursley's sister after sh...,"chunk_20, chunk_27, chunk_2125",It is not mentioned what happened to Mr. Dursl...,False
7,Why did Hermione warn Harry not to wander arou...,"chunk_2607, chunk_422, chunk_1833","I don't know.\n\n[chunk_2607, Harry Potter and...",False
8,Where did Hagrid keep the large black egg?,"chunk_756, chunk_7732, chunk_266",Hagrid kept the large black egg in his coat.\n...,True
9,What was happening to the car as it flew towar...,"chunk_1235, chunk_1239, chunk_9635",The car was losing speed and was experiencing ...,True


**Retrieval Evaluation Results**
- The retriever was tested on 10 sample questions using top_k = 3.
- The true chunk was successfully retrieved for 6 out of 10 questions, resulting in an overall recall of 60%, but the retrievel failed for 4 questions
- there was wrong answers like *Where did Harry sleep?* The answer was grounded because the correct chunk (chunk_61) was not retrieved.it maybe beacouse the question is too general and lacks context to distinguish between different events where Harry was sleeping.
- Question: *What happened to Mr. Dursley's sister after she received a letter* The answer was not hallucinated because the retrieved context did not contain enough information to answer the question. The model correctly responded, “I don't know.”
- To mitigate these issues, increasing the number of retrieved chunks can improve the chance of retrieving the true chunk.